# PCA scaling analysis — Replogle

How many principal components are needed to **saturate** linear reconstruction of gene expression (2000 HVGs)?

- **Fit:** `sc.tl.pca` on `split2 ∈ {train, finetune}` (same as `analyze_and_preprocess_replogle.ipynb`)
- **Evaluate:** Pearson and MSE reconstruction on **all** cells (train / finetune / val / test)
- **Projection / decode:** `(X − mean_train) @ PCs` and `z @ PCs.T + mean_train`

Plots: mean per-cell Pearson and mean per-cell MSE vs `n_pcs`; by-split panel with Pearson and MSE side by side.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

REPO = Path("../../").resolve()
H5AD_PATH = REPO / "data_dir/replogle_with_splits.h5ad"
SPLIT_COL = "split2"
FIT_SPLITS = ("train", "finetune")
N_PCS_GRID = (50, 100, 200, 300, 500, 1000)
N_COMPS_FIT = max(N_PCS_GRID)
BATCH_SIZE = 8192
RANDOM_SEED = 0
OUT_DIR = REPO / "results_dir/pca_scaling/replogle"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

In [ ]:
H5AD_PATH

In [ ]:
def read_X_batch(adata, row_idx: np.ndarray) -> np.ndarray:
    chunk = adata.X[row_idx]
    if hasattr(chunk, "toarray"):
        chunk = chunk.toarray()
    return np.asarray(chunk, dtype=np.float64)


def pearson_per_cell(true: np.ndarray, recon: np.ndarray) -> np.ndarray:
    t = true - true.mean(axis=1, keepdims=True)
    r = recon - recon.mean(axis=1, keepdims=True)
    num = (t * r).sum(axis=1)
    den = np.sqrt((t * t).sum(axis=1) * (r * r).sum(axis=1))
    return num / np.maximum(den, 1e-12)


def evaluate_reconstruction(
    adata,
    row_idx: np.ndarray,
    mean: np.ndarray,
    pcs: np.ndarray,
    n_pcs: int,
    *,
    batch_size: int,
) -> dict[str, float]:
    loadings = pcs[:, :n_pcs]
    pearson_chunks: list[np.ndarray] = []
    mse_chunks: list[np.ndarray] = []
    for start in range(0, len(row_idx), batch_size):
        chunk = read_X_batch(adata, row_idx[start : start + batch_size])
        z = (chunk - mean) @ loadings
        recon = z @ loadings.T + mean
        pearson_chunks.append(pearson_per_cell(chunk, recon))
        mse_chunks.append(((chunk - recon) ** 2).mean(axis=1))
    pearson = np.concatenate(pearson_chunks)
    mse = np.concatenate(mse_chunks)
    return {
        "mean_pearson": float(pearson.mean()),
        "median_pearson": float(np.median(pearson)),
        "mean_mse": float(mse.mean()),
        "median_mse": float(np.median(mse)),
    }

In [ ]:
import anndata as ad

adata = ad.read_h5ad(H5AD_PATH)
print(f"Loaded {H5AD_PATH.name}: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(adata.obs[SPLIT_COL].value_counts())

fit_mask = adata.obs[SPLIT_COL].isin(FIT_SPLITS).to_numpy()
fit_idx = np.where(fit_mask)[0]
all_idx = np.arange(adata.n_obs)
print(f"PCA fit cells: {len(fit_idx):,}")

In [ ]:
# Match analyze_and_preprocess_replogle.ipynb: sc.tl.pca on train+finetune only.
adata_train_ft = adata[adata.obs[SPLIT_COL].isin(FIT_SPLITS)].copy()
print(f"Fitting sc.tl.pca on {adata_train_ft.n_obs:,} cells with n_comps={N_COMPS_FIT}...")
sc.tl.pca(adata_train_ft, n_comps=N_COMPS_FIT)

pcs = np.asarray(adata_train_ft.varm["PCs"], dtype=np.float64)
mean_train = np.asarray(adata_train_ft.X.mean(0), dtype=np.float64).ravel()
print(f"PC loadings shape: {pcs.shape}")

In [ ]:
rows: list[dict] = []
for n_pcs in N_PCS_GRID:
    print(f"Evaluating n_pcs={n_pcs} on all {len(all_idx):,} cells...")
    metrics = evaluate_reconstruction(
        adata, all_idx, mean_train, pcs, n_pcs, batch_size=BATCH_SIZE
    )
    rows.append({"n_pcs": n_pcs, "scope": "all", **metrics})

    for split in adata.obs[SPLIT_COL].cat.categories:
        split_idx = np.where((adata.obs[SPLIT_COL] == split).to_numpy())[0]
        split_metrics = evaluate_reconstruction(
            adata, split_idx, mean_train, pcs, n_pcs, batch_size=BATCH_SIZE
        )
        rows.append({"n_pcs": n_pcs, "scope": split, **split_metrics})

results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / "pca_scaling_metrics.csv", index=False)
results.query('scope == "all"')

In [ ]:
summary = results.query('scope == "all"').sort_values("n_pcs")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(summary["n_pcs"], summary["mean_pearson"], marker="o", linewidth=2)
axes[0].set_xlabel("n_pcs")
axes[0].set_ylabel("Mean per-cell Pearson r")
axes[0].set_title("Replogle — Pearson reconstruction")
axes[0].set_xticks(list(N_PCS_GRID))

axes[1].plot(summary["n_pcs"], summary["mean_mse"], marker="o", linewidth=2, color="C1")
axes[1].set_xlabel("n_pcs")
axes[1].set_ylabel("Mean per-cell MSE")
axes[1].set_title("Replogle — MSE reconstruction")
axes[1].set_xticks(list(N_PCS_GRID))

fig.tight_layout()
fig.savefig(OUT_DIR / "pca_scaling_replogle.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for split in adata.obs[SPLIT_COL].cat.categories:
    sub = results.query('scope == @split').sort_values("n_pcs")
    axes[0].plot(sub["n_pcs"], sub["mean_pearson"], marker="o", label=split)
    axes[1].plot(sub["n_pcs"], sub["mean_mse"], marker="o", label=split)

axes[0].set_xlabel("n_pcs")
axes[0].set_ylabel("Mean per-cell Pearson r")
axes[0].set_title("Replogle — Pearson by split")
axes[0].set_xticks(list(N_PCS_GRID))
axes[0].legend(title=SPLIT_COL)

axes[1].set_xlabel("n_pcs")
axes[1].set_ylabel("Mean per-cell MSE")
axes[1].set_title("Replogle — MSE by split")
axes[1].set_xticks(list(N_PCS_GRID))
axes[1].legend(title=SPLIT_COL)

fig.tight_layout()
fig.savefig(OUT_DIR / "pca_scaling_replogle_by_split.png", bbox_inches="tight")
plt.show()